# Preprocessing & EDA
## Healthcare Sector Focus

**Goal**: Analyse the UK ICO data security incident trends with a focus on the healthcare sector to prepare for building ML models and a dashboard.


In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay, silhouette_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from kmodes.kmodes import KModes

In [ ]:
# load the dataset
df_ico_original = pd.read_csv("data-security-incidents-trends-q1-2019-to-q4-2025.csv", delimiter = ",")

# initial look at data
df_ico_original.info()

In [ ]:
# count all sectors
sector_counts = df_ico_original['Sector'].value_counts()
sector_percents = sector_counts / len(df_ico_original) * 100

plt.figure(figsize=(10, 8))
bars = plt.bar(sector_counts.index, sector_counts.values)

plt.title("Data Security Incidents by Sector")
plt.xlabel("Sector")
plt.ylabel("Incident Count")
plt.xticks(rotation=45, ha='right')
plt.yticks(np.arange(0, max(sector_counts.values) + 1, 2500))

# add percentage labels on top of each bar
for bar, percent in zip(bars, sector_percents):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2, # put percentage in the middle
        height, f"{percent:.1f}%",
        ha='center', va='bottom'
    )

print(f"\nTotal sectors: {df_ico_original['Sector'].nunique()}")
plt.grid(True, axis='y', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Sector table with counts and percentages
sector_table = pd.DataFrame({
    'Incident Count': sector_counts,
    'Percentage': sector_percents.round(3)
})
sector_table


In [ ]:
# filtering the database to focus of the health sector
df_health_original = df_ico_original[df_ico_original["Sector"] == "Health"].copy(deep=True)

# drop the sector column as it's redundant now
df_health_original = df_health_original.drop(columns=["Sector"])

# standardising the column names
df_health_original.columns = df_health_original.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(".", "")

# look at new healthcare sector focused df
df_health_original.info()
#df_health_original.to_csv("data-security-incident-trends-health-sector.csv", index=False)

In [ ]:
# filtering timeframe to Q2 2021 to Q2 2025

# Get quarters as int for filtering
df_health_original["quarter_num"] = df_health_original["quarter"].str.extract(r"(\d)").astype(int) # temp, helper column 

df_health_original = df_health_original[
    ((df_health_original["year"] > 2021) | ((df_health_original["year"] == 2021) & (df_health_original["quarter_num"] >= 2))) &
    ((df_health_original["year"] < 2025) | ((df_health_original["year"] == 2025) & (df_health_original["quarter_num"] <= 4)))
]

df_health_original = df_health_original.drop(columns="quarter_num") # drop 

# Test to see full timeframe
print(df_health_original[["year", "quarter"]].drop_duplicates().sort_values(["year", "quarter"]).to_string(index=False))

# New CSV of all processed data
#df_health_original.to_csv("data-security-incident-trends-health-sector.csv", index=False)

In [ ]:
# change data type of features into categorical instead of object/ int as characteristics are fixed
categorical_cols = [
    "bi_reference",
    "year",
    "quarter",
    "data_subject_type",
    "data_type",
    "decision_taken",
    "incident_category",
    "incident_type",
    "no_data_subjects_affected",
    "time_taken_to_report"
]


for col in categorical_cols:
    df_health_original[col] = df_health_original[col].astype("category")

# verify new data types
df_health_original.info()

In [ ]:
# Order the data subjects affected category
df_health_original["no_data_subjects_affected"] = pd.Categorical(
    df_health_original["no_data_subjects_affected"],
    categories=[
        "1 to 9",
        "10 to 99",
        "100 to 1k",
        "1k to 10k",
        "10k to 100k",
        "100k and above",
        "Unknown"
    ],
    ordered=True
)

# verify ordering
df_health_original["no_data_subjects_affected"].cat.categories

In [ ]:
# finding the unique vs total BI references 
total_rows = len(df_health_original)
unique_bi_refs = df_health_original['bi_reference'].nunique()
duplicate_ratio = (total_rows - unique_bi_refs) / total_rows * 100

print(f"\nTotal rows in the dataset: {total_rows:,}")
print(f"Unique BI References: {unique_bi_refs:,}")
print(f"Rows representing duplicate characteristics: {total_rows - unique_bi_refs:,}")
print(f"Duplication rate: {duplicate_ratio:.2f}%")

In [ ]:
# plot: pie chart of unique vs duplicate references
bi_ref_counts = df_health_original['bi_reference'].value_counts()
single_row = (bi_ref_counts == 1).sum()
multiple_rows = (bi_ref_counts > 1).sum()

plt.pie([single_row, multiple_rows], 
            labels=['Single Row\n(No duplication)', 'Multiple Rows\n(Duplicated)'], 
            autopct='%1.1f%%',
            colors=['#ff9999', '#66b3ff'],
            startangle=90,
            textprops={'fontsize': 11})

plt.title('BI References (Health Sector): Single vs Multiple Rows', fontsize=12, fontdict={'weight': 'bold'})
plt.show()

In [ ]:
# analyse which columns typically vary across duplicates
duplicated_bi_refs = df_health_original[df_health_original['bi_reference'].duplicated(keep=False)]
total_groups = duplicated_bi_refs['bi_reference'].nunique()

# nunique per group per column
varying_columns = duplicated_bi_refs.groupby("bi_reference", observed=False).nunique()

# a column varies in a group if nunique > 1
varying_counts = (varying_columns > 1).sum()

varying_counts = (varying_counts
    .sort_values(ascending=False)
    .rename("count")
    .to_frame()
)

varying_counts["percentage"] = (varying_counts["count"] / total_groups * 100)

print(f"Columns that vary across duplicate rows:\n {varying_counts}")

# summarise how many characteristics each breach has
print("\nBreach characteristics per incident:")
df_health_original.groupby("bi_reference", observed=False).size().describe()

In [ ]:
# checking how many missing/ unknown values are in the healthcare df
print(f"Missing values in the healthcare dataset:\n{df_health_original.isnull().sum()}")

unknown_counts = df_health_original[categorical_cols].isin(["Unknown"]).sum().sort_values(ascending=False)

# calculate percentages
unknown_percents = (unknown_counts / len(df_health_original)) * 100

# visualise per feature
plt.figure(figsize=(10, 6))
bars = plt.bar(unknown_counts.index, unknown_counts.values)

plt.title("Unknown Values by Feature (Count and Percentage)")
plt.xlabel("Feature")
plt.ylabel("Number of Rows")
plt.xticks(rotation=45, ha="right")

# add percentage labels
for bar, percent in zip(bars, unknown_percents):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height, f"{percent:.1f}%",
        ha="center", va="bottom"
    )

plt.grid(True, axis='y', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()
print(f"Unknown characteristic counts in each feature:\n{unknown_counts}")

In [ ]:
# Create a new df that only contains unique BI references (deduplicated)
df_health_unique = df_health_original.drop_duplicates(subset=["bi_reference"], keep="first").reset_index(drop=True)
print("Unique Rows in Health Sector df:",unique_bi_refs)

In [ ]:
# aggregate total rows per year-quarter (duplicated BI references)
total_incidents = (
    df_health_original
    .groupby(["year", "quarter"], observed=True) 
    .size()
    .reset_index(name="total_incidents")
)

# aggregate unique incidents per year-quarter (non-duplicated BI references)
unique_incidents = (
    df_health_unique
    .groupby(["year", "quarter"], observed=True)  
    .size()
    .reset_index(name="unique_incidents")
)

quarters = {"Qtr 1": 1, "Qtr 2": 2, "Qtr 3": 3, "Qtr 4": 4}

# merge the two summaries
incidents_over_time = total_incidents.merge(unique_incidents, on=["year", "quarter"])

# make the chronological order
incidents_over_time["quarter_num"] = incidents_over_time["quarter"].astype(str).map(quarters)

# convert year to int for sorting
incidents_over_time["year_num"] = incidents_over_time["year"].astype(str).astype(int)
incidents_over_time = incidents_over_time.sort_values(by=["year_num", "quarter_num"]).reset_index(drop=True)

# x-axis label
incidents_over_time["period"] = (incidents_over_time["year"].astype(str) + " " + incidents_over_time["quarter"].astype(str))

# copy for display purposes
display_df = incidents_over_time[["year", "quarter", "total_incidents", "unique_incidents"]].copy()
# calculate the difference (duplicate count) & percentage of duplicates
display_df["duplicates"] = display_df["total_incidents"] - display_df["unique_incidents"]
# display columns
display_df.columns = ["Year", "Quarter", "Total Rows", "Unique Incidents", "Duplicates"]

# tabular form
print("Healthcare data security incidents over time:")
print(display_df.to_string(index=False))

# summary statistics
print(f"\nOverall Summary:")
print(f"\tTotal rows across all periods: {display_df['Total Rows'].sum():,}")
print(f"\tTotal unique incidents: {display_df['Unique Incidents'].sum():,}")
print(f"\tTotal duplicate incidents: {display_df['Duplicates'].sum():,}")

# mean and standard deviations of unique incidents over time 
print(f"\tMean incidents per quarter: {display_df['Unique Incidents'].mean():.1f}")
print(f"\tStandard deviation of incidents: {display_df['Unique Incidents'].std():.1f}")
print(f"\tMaximum incidents in a quarter: {display_df['Unique Incidents'].max():,}")
print(f"\tMinimum incidents in a quarter: {display_df['Unique Incidents'].min():,}")

In [ ]:
# graph visualisation of incidents 
plt.figure(figsize=(12, 6))

plt.plot(
    incidents_over_time.index,
    incidents_over_time["total_incidents"],
    marker='o',
    label="All reported rows (duplicated BI references)",
    linestyle="--",
    linewidth=2,
    markersize=6
)

plt.plot(
    incidents_over_time.index,
    incidents_over_time["unique_incidents"],
    marker='s',
    label="Unique incidents (non-duplicated BI references)",
    linewidth=2,
    markersize=6
)

plt.xticks(
    ticks=incidents_over_time.index,
    labels=incidents_over_time["period"],
    rotation=45, ha='right'
)

plt.title("Healthcare Data Security Incidents Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Year and Quarter", fontsize=12)
plt.ylabel("Incident Count", fontsize=12)
plt.legend(loc='best', frameon=True)
plt.grid(True, linestyle='-')
plt.tight_layout()
plt.show()

In [ ]:
# cyber vs non-cyber incidents
incident_category_counts = df_health_unique['incident_category'].value_counts()
incident_category_percents = (incident_category_counts / len(df_health_original)) * 100

plt.title("Cyber vs Non-Cyber Incidents (Health Sector)", fontsize=12, fontweight='bold')
plt.pie(incident_category_counts.values,
        labels=incident_category_counts.index,
        autopct='%1.1f%%',
        colors=["#ffd699", "#66b3ff"],
        startangle=90,
        textprops={'fontsize': 11})

plt.tight_layout()
plt.show()
print(f"Incident category counts (Cyber vs Non-cyber):\nCyber: {incident_category_counts['Cyber']:,} \nNon-Cyber: {incident_category_counts['Non Cyber']:,}")

In [ ]:
# decision taken outcomes
decision_counts = df_health_unique['decision_taken'].value_counts()
decision_percents = (decision_counts / len(df_health_original)) * 100

# pie chart
plt.title("Decision Taken Outcomes (Health Sector)", fontsize=14, fontweight='bold')
plt.pie(decision_counts.values,
        labels=decision_counts.index,
        autopct='%1.1f%%',
        colors=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'],
        startangle=90,
        textprops={'fontsize': 10})

plt.tight_layout()
plt.show()

print(f"Decision taken counts:\nInformal Action Taken:\t{decision_counts['Informal Action Taken']:,} \nNo Further Action:\t{decision_counts['No Further Action']:,} \nInvestigation Pursued:\t{decision_counts['Investigation Pursued']:,} \nNot Yet Assigned:\t{decision_counts['Not Yet Assigned']:,}")


In [ ]:
# incident type vs data subjects affected

# proportions
incident_severity_proportions = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['no_data_subjects_affected'],
    normalize='index'
)
# counts
incident_severity_counts = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['no_data_subjects_affected']
)
# horizontal bar chart
incident_severity_proportions.plot(
    kind='barh',
    stacked=True,
    figsize=(16, 8),
    colormap='plasma'
)

plt.title('Incident Type vs Proportion of People Affected', fontsize=16, fontweight='bold')
plt.ylabel('Incident Type', fontsize=14)
plt.xlabel('Proportion of People Affected', fontsize=14)
plt.xticks(np.arange(0, 1.1, 0.1), [f"{int(x*100)}%" for x in np.arange(0, 1.1, 0.1)])
plt.legend(title='Number of People Affected', bbox_to_anchor=(1.05, 1), loc='best')
plt.grid(True, axis='x', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()

# display the incident type vs data subjects affected proportions
incident_severity_proportions.round(3)

In [ ]:
# display the incident type vs data subjects affected counts
incident_severity_counts

In [ ]:
# decision taken vs incident Type

# proportions
incident_type_decision_proportions = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['decision_taken'],
    normalize='index'
)
# counts
incident_type_decision_counts = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['decision_taken']
)
# horizontal bar chart
incident_type_decision_proportions.plot(
    kind='barh',
    stacked=True,
    figsize=(16, 8),
    colormap='plasma'
)
plt.title('Incident Type vs Decision Taken Outcomes', fontsize=16, fontweight='bold')
plt.xlabel('Proportion of Decisions Taken', fontsize=14)
plt.ylabel('Incident Type', fontsize=14)
plt.xticks(np.arange(0, 1.1, 0.1), [f"{int(x*100)}%" for x in np.arange(0, 1.1, 0.1)])
plt.legend(title='Decision Taken', bbox_to_anchor=(1.05, 1), loc='best')
plt.grid(True, axis='x', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()

# table for the incident type vs decision taken proportions
incident_type_decision_proportions.round(3)

In [ ]:
# table for the incident type vs decision taken counts
incident_type_decision_counts

In [ ]:
# decision taken vs no. data subjects affected

# proportions
decision_severity_proportions = pd.crosstab(
    df_health_unique['no_data_subjects_affected'],
    df_health_unique['decision_taken'],
    normalize='index'
)
# counts
decision_severity_counts = pd.crosstab(
    df_health_unique['no_data_subjects_affected'],
    df_health_unique['decision_taken']
)
# bar chart
decision_severity_proportions.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6),
    colormap='plasma'
)

plt.title('Number of People Affected vs Decision Taken Outcomes', fontsize=16, fontweight='bold')
plt.xlabel('Number of People Affected', fontsize=14)
plt.xticks(rotation=0, ha='center')
plt.ylabel('Proportion of Decisions Taken', fontsize=14)
plt.yticks(np.arange(0, 1.1, 0.1), [f"{int(x*100)}%" for x in np.arange(0, 1.1, 0.1)])
plt.legend(title='Decision Taken', bbox_to_anchor=(1.05, 1), loc='best')
plt.grid(True, axis='y', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()

# table for the number of people affected vs decision taken proportions
decision_severity_proportions.round(3)


In [ ]:
# table for the number of people affected vs decision taken counts
decision_severity_counts

In [ ]:
# incident category vs incident Type

# proportions
incident_type_category_proportions = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['incident_category'],
    normalize='index'
)
# counts
incident_type_category_counts = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['incident_category']
)
# horizontal bar chart
incident_type_category_proportions.plot(
    kind='barh',
    stacked=True,
    figsize=(12, 6),
    colormap='plasma'
)

plt.title('Incident Type vs Incident Category', fontsize=16, fontweight='bold')
plt.ylabel('Incident Type', fontsize=14)
plt.xlabel('Count of Incident Categories', fontsize=14)
plt.yticks(rotation=0, ha='right')
plt.xticks(np.arange(0, 1.1, 0.1), [f"{int(x*100)}%" for x in np.arange(0, 1.1, 0.1)]) 
plt.legend(title='Incident Category', bbox_to_anchor=(1.05, 1), loc='best')
plt.grid(True, axis='x', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()

# table for the incident type vs incident category proportions
incident_type_category_proportions.round(3)


In [ ]:
# table for the incident type vs incident category counts
incident_type_category_counts

In [ ]:
# incident type vs time taken to report

# proportions
incident_type_time_proportions = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['time_taken_to_report'],
    normalize='index'
)
# counts
incident_type_time_counts = pd.crosstab(
    df_health_unique['incident_type'],
    df_health_unique['time_taken_to_report']
)
# horizontal bar chart
incident_type_time_proportions.plot(
    kind='barh',
    stacked=True,
    figsize=(16, 8),
    colormap='plasma'
)

plt.title('Incident Type vs Time Taken to Report', fontsize=16, fontweight='bold')
plt.ylabel('Incident Type', fontsize=14)
plt.xlabel('Time Taken to Report', fontsize=14)
plt.yticks(rotation=0, ha='right')
plt.xticks(np.arange(0, 1.1, 0.1), [f"{int(x*100)}%" for x in np.arange(0, 1.1, 0.1)]) 
plt.legend(title='Time Taken to Report', bbox_to_anchor=(1.05, 1), loc='best')
plt.grid(True, axis='x', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()

# table for the incident type vs time taken to report proportions
incident_type_time_proportions.round(3)

In [ ]:
# table for the incident type vs time taken to report counts
incident_type_time_counts

In [ ]:
# decision taken vs time taken to report

# proportions
decision_time_proportions = pd.crosstab(
    df_health_unique['time_taken_to_report'],
    df_health_unique['decision_taken'],
    normalize='index'
)
# counts
decision_time_counts = pd.crosstab(
    df_health_unique['time_taken_to_report'],
    df_health_unique['decision_taken']
)
# bar chart
decision_time_proportions.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6),
    colormap='plasma'
)

plt.title('Time Taken to Report vs Decision Taken Outcomes', fontsize=16, fontweight='bold')
plt.xlabel('Time Taken to Report', fontsize=14)
plt.xticks(rotation=0, ha='center')
plt.ylabel('Proportion of Decisions Taken', fontsize=14)
plt.yticks(np.arange(0, 1.1, 0.1), [f"{int(x*100)}%" for x in np.arange(0, 1.1, 0.1)])
plt.legend(title='Decision Taken', bbox_to_anchor=(1.05, 1), loc='best')
plt.grid(True, axis='y', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()     

# table for the time taken to report vs decision taken proportions
decision_time_proportions.round(3)

In [ ]:
# table for the time taken to report vs decision taken counts
decision_time_counts

In [ ]:
# proportion of number of people affected across all incidents
data_subjects_counts = df_health_unique['no_data_subjects_affected'].value_counts().sort_index()
data_subjects_percents = (data_subjects_counts / len(df_health_unique)) * 100
print(f"\nNumber of People Affected across all Healthcare Incidents:")
for category, count, percent in zip(data_subjects_counts.index, data_subjects_counts.values, data_subjects_percents.values):
    print(f"\t{category}: {count:,} incidents ({percent:.1f}%)")
    

# pie chart of number of people affected
plt.title("Number of People Affected (Health Sector)", fontsize=12, fontweight='bold')
plt.pie(data_subjects_counts.values,
        labels=data_subjects_counts.index,
        autopct='%1.1f%%',
        colors=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'],
        startangle=90,
        textprops={'fontsize': 11})
plt.tight_layout()
plt.show()

# proportion of incident type across all incidents
incident_type_counts = df_health_unique['incident_type'].value_counts()
incident_type_percents = (incident_type_counts / len(df_health_unique)) * 100
print(f"\nIncident Types across all Healthcare Incidents:")
for category, count, percent in zip(incident_type_counts.index, incident_type_counts.values, incident_type_percents.values):
    print(f"\t{category}: {count:,} incidents ({percent:.1f}%)")
    
# pie chart of incident types
plt.title("Incident Types (Health Sector)", fontsize=12, fontweight='bold')
plt.pie(incident_type_counts.values,
        labels=incident_type_counts.index,
        autopct='%1.1f%%',
        colors=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'],
        startangle=90,
        textprops={'fontsize': 11})
plt.tight_layout()
plt.show()

# proportion of time taken to report across all incidents
time_taken_counts = df_health_unique['time_taken_to_report'].value_counts()
time_taken_percents = (time_taken_counts / len(df_health_unique)) * 100
print(f"\nTime Taken to Report across all Healthcare Incidents:")
for category, count, percent in zip(time_taken_counts.index, time_taken_counts.values, time_taken_percents.values):
    print(f"\t{category}: {count:,} incidents ({percent:.1f}%)")
    
# pie chart of time taken to report
plt.title("Time Taken to Report (Health Sector)", fontsize=12, fontweight='bold')
plt.pie(time_taken_counts.values,
        labels=time_taken_counts.index,
        autopct='%1.1f%%',
        colors=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'],
        startangle=90,
        textprops={'fontsize': 11})
plt.tight_layout()
plt.show()

In [ ]:
# make the df_health_original df a new csv file for the next stages of analysis
# check the columns and data types before saving
df_health_original.info()
#df_health_original.to_csv("data-security-incident-trends-health-sector.csv", index=False)

In [ ]:
# create a new df for ML models that includes unique incidents with aggregated data_subject_type and data_type features
df_health_ml = (
    df_health_original
    .groupby("bi_reference", observed=True)
    .agg({
        "year": "first",
        "quarter": "first",
        "data_subject_type": lambda data: ', '.join(sorted(data.unique())),
        "data_type": lambda data: ', '.join(sorted(data.unique())),
        "decision_taken": "first",
        "incident_category": "first",
        "incident_type": "first",
        "no_data_subjects_affected": "first",
        "time_taken_to_report": "first",
    })
    .reset_index()
)

# drop bi_reference as not needed for modelling
df_health_ml = df_health_ml.drop(columns=["bi_reference"])

# drop the 'Not Yet Assigned' rows as they aren't predictable 
df_health_ml = df_health_ml[df_health_ml["decision_taken"] != "Not Yet Assigned"].reset_index(drop=True)

# verify new df
df_health_ml.info()
df_health_ml.to_csv("new-data-security-incident-trends-health-sector.csv", index=False)

In [ ]:
# check decision taken variables 
df_health_ml["decision_taken"].value_counts()

In [ ]:
# Decision outcomes over time
decision_time_summary = (
    df_health_ml
    .groupby(["year", "quarter"], observed=True)
    ["decision_taken"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# plot the decision outcomes over time
plt.figure(figsize=(12, 6))
for decision in decision_time_summary["decision_taken"].unique():
    subset = decision_time_summary[decision_time_summary["decision_taken"] == decision]
    plt.plot(
        subset.index,
        subset["proportion"],
        marker='o',
        label=decision,
        linewidth=2,
        markersize=6
    ) 
quarters = {"Qtr 1": 1, "Qtr 2": 2, "Qtr 3": 3, "Qtr 4": 4}
decision_time_summary["quarter_num"] = decision_time_summary["quarter"].astype(str).map(
    quarters
)
decision_time_summary["year_num"] = decision_time_summary["year"].astype(str).astype(int)
decision_time_summary = decision_time_summary.sort_values(by=["year_num", "quarter_num"]).reset_index(drop=True)
decision_time_summary["period"] = (decision_time_summary["year_num"].astype(str) + " Q" + decision_time_summary["quarter_num"].astype(str))
plt.xticks(
    ticks=decision_time_summary.index,
    labels=decision_time_summary["period"],
    rotation=45, ha='right'
)
plt.title("Proportion of Decision Outcomes Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Year and Quarter", fontsize=12)
plt.ylabel("Proportion of Incidents", fontsize=12)

plt.legend(loc='best', frameon=True)
plt.grid(True, linestyle='-')
plt.tight_layout()
plt.show()